# Notebook Objectives

In this notebooks, we'll go through the main behavioral analysis results.

Results include:
* ?
* ?

---
# Setup


In [1]:
from neural_value_helpers import *

In [2]:
##% data loading
def load_data_custom(monkey, area=None, subregion=None):
    # load data (meta session)
    floc = os.path.join(PATH, 'data', 'processed', 'neural_data', 'meta_rates_history')
    neural_dataset =  xr.open_dataset(floc + '/meta_rates.nc')

    if area is not None:
        neural_dataset = neural_dataset.sel(unit=neural_dataset.area == area)
    if subregion is not None:
        neural_dataset = neural_dataset.sel(unit=neural_dataset.subregion == subregion)

    neural_dataset = neural_dataset.sel(unit=neural_dataset.monkey == monkey)

    # add coordinates (should be added during the dataset creation)
    neural_dataset = neural_dataset.assign_coords(R_1=('trial_id', [0 if hist in [0, 1, 2, 3] else 1 for hist in neural_dataset.history_of_feedback.values]))
    neural_dataset = neural_dataset.assign_coords(R_2=('trial_id', [0 if hist in [0, 1, 4, 5] else 1 for hist in neural_dataset.history_of_feedback.values]))
    neural_dataset = neural_dataset.assign_coords(R_3=('trial_id', [0 if hist in [0, 2, 4, 6] else 1 for hist in neural_dataset.history_of_feedback.values]))
    neural_dataset = neural_dataset.assign_coords(fb_sequence=('trial_id', neural_dataset.history_of_feedback.values))
    neural_dataset = neural_dataset.assign_coords(fb_sequence_m1=('trial_id', neural_dataset.history_of_feedback.values%4))
    neural_dataset = neural_dataset.assign_coords(time_m1=('time', neural_dataset.time.values + 7.5))

    # normalize firing rates (z-score) and remove units with low firing rates
    threshold = 1  # Hz
    units_to_remove = []
    normalized_data = np.zeros_like(neural_dataset.firing_rates.data)  # Create a copy to avoid modifying the original data
    for i, unit in enumerate(neural_dataset.unit.values):  # Loop through each neuron
        # Get all data for this neuron across trials and timepoints
        unit_data = neural_dataset.firing_rates.sel(unit=unit).values
        
        # Calculate mean and std across all values for this neuron
        mean_val = np.mean(unit_data)
        std_val = np.std(unit_data)
        
        # Z-score normalize and store in the output array
        if mean_val < threshold:
            units_to_remove.append(unit)
        elif std_val == 0:
            raise ValueError(f"Standard deviation is zero for unit {unit}. Not possible.")
        else:
            z_scored_data = (unit_data - mean_val) / std_val        
            normalized_data[:, i, :] = z_scored_data
    #neural_dataset.firing_rates.values = normalized_data
    neural_dataset['firing_rates'] = (neural_dataset.firing_rates.dims, normalized_data)

    neural_dataset = neural_dataset.sel(unit=~neural_dataset.unit.isin(units_to_remove))
    print(f"Removed {len(units_to_remove)} units with mean firing rate < {threshold} Hz")

    return neural_dataset

---
# Get data

In [3]:
monkey, subregion = 'ka', 'MCC'

t_interest_fit = -7.5  # time point of interest for projection
t_interest_value = [2.5, 3.5]

neural_dataset = load_data_custom(monkey, subregion=subregion)
#neural_dataset = neural_dataset.sel(time=slice(0, 6))

Removed 85 units with mean firing rate < 1 Hz


---
# Section 1: Projection to value subspace

## Define the value subspace

In [4]:
floc = f'data/data_projected_temp_{monkey}_{subregion}_{str(t_interest_fit)}.nc'

if not os.path.exists(floc):
    results, weights, data_projected, pvalue = time_resolved_decoder(neural_dataset, target='R_1', t_project=t_interest_fit)

    print(pvalue, pvalue < 0.05)

    # save data_projected
    data_projected.to_netcdf(floc)

    fig, axs = plot_decoder_results(results, weights, n_extra_trials=(-1, 0))

    plt.suptitle(f'{monkey} {subregion} - decoding R_1')
    plt.tight_layout()
else:
    data_projected = xr.open_dataarray(floc)


KeyboardInterrupt: 

In [ ]:
'''units = neural_dataset.unit.values
# bet best weights id
best_weights = weights.sel(time=3.5)
# sort by absolute value
best_weights = np.argsort(np.abs(best_weights.values).squeeze())[::-1]
best_unit = units[best_weights]
print(f'Best unit: {best_unit}')'''

In [ ]:
# normalize (mean substract)
data_projected = data_projected - data_projected.mean(dim='trial_id')

In [ ]:
fig, ax = plot_projected_data(data_projected, t_interest_fit, t_interest_value, n_extra_trials=(-1, 0), xlim=None, paper_format=False)
plt.suptitle(f'{monkey} {subregion}, t_fit = {t_interest_fit}s, t_eval = {t_interest_value}s')

# save figure
#fig.savefig(f'figs/projected_{monkey}_{subregion}.svg', bbox_inches='tight', dpi=300, transparent=True)

---
# Section 2: Analysis of the neural value representation

## Green red plot: Value evolves according to feedback

In [ ]:
data_projected = add_neural_value_coord(data_projected, t_interest=t_interest_value)
data_projected

In [ ]:
fig, ax = projection_timepoint(data_projected, paper_format=False, ylim=None)
plt.suptitle(f'{monkey} {subregion}, t_fit = {t_interest_fit}s, t_eval = {t_interest_value}s')

#fig.savefig(f'figs/projection_detailed_{monkey}_{subregion}.svg', bbox_inches='tight', transparent=True, dpi=300)

In [ ]:
fig, ax = green_red_plot(data_projected, paper_format=False, xlim=None, ylim=None)
plt.suptitle(f'{monkey} {subregion}, t_fit = {t_interest_fit}s, t_eval = {t_interest_value}s,')

#fig.savefig(f'figs/green_red_plot_{monkey}_{subregion}.svg', bbox_inches='tight', transparent=True, dpi=300)